# GPA Predictor — End-to-End Notebook (CSV-only)

This notebook covers:
- Loading data **only** from `data/mock_gpa.csv`
- Basic cleaning / checks
- Train/test split
- Model training (Linear Regression)
- Evaluation (MAE, RMSE, R²)
- Saving model to `models/gpa_model.pkl`

Run from the repository root so relative paths work.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "mock_gpa.csv"
MODEL_DIR = BASE_DIR / "models"
MODEL_PATH = MODEL_DIR / "gpa_model.pkl"

BASE_DIR, DATA_PATH, MODEL_PATH

## 1) Load data (CSV-only)

This notebook requires `data/mock_gpa.csv`. If the file is missing, it will stop with an error.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Required dataset not found: {DATA_PATH}. "
        "Please add data/mock_gpa.csv before running this notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {DATA_PATH}")
df.head()

### Validate required columns

In [ ]:
required_cols = {"study_hours", "IQ", "attendance", "sleep_hours", "GPA"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

# Keep only required columns (optional but helps avoid surprises)
df = df[["study_hours", "IQ", "attendance", "sleep_hours", "GPA"]].copy()
df.head()

## 2) Quick data checks

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

## 3) Prepare features/target

Features:
- `study_hours`, `IQ`, `attendance`, `sleep_hours`

Target:
- `GPA`

In [ ]:
X = df[["study_hours", "IQ", "attendance", "sleep_hours"]]
y = df["GPA"]

X.head(), y.head()

## 4) Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

## 5) Train model (Linear Regression)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

model.coef_, model.intercept_

## 6) Evaluate

In [ ]:
y_pred = model.predict(X_test)
y_pred_clipped = np.clip(y_pred, 0, 4)

mae = mean_absolute_error(y_test, y_pred_clipped)
rmse = mean_squared_error(y_test, y_pred_clipped, squared=False)
r2 = r2_score(y_test, y_pred_clipped)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")

## 7) Save the trained model

This writes the model to `models/gpa_model.pkl` (the path used by your FastAPI app).

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Saved model to: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size} bytes")

## 8) Quick inference test (matches FastAPI input fields)

In [ ]:
sample = {
    "study_hours": 20,
    "IQ": 110,
    "attendance": 85,
    "sleep_hours": 7,
}

x = np.array([
    sample["study_hours"],
    sample["IQ"],
    sample["attendance"],
    sample["sleep_hours"],
]).reshape(1, -1)

pred = float(np.clip(model.predict(x)[0], 0, 4))
pred